In [1]:
!pip install -q ultralytics #lap onnxruntime onnxslim
#!pip install tensorrt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 77.4 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires nu

In [3]:
import shutil

shutil.copytree(
    "/kaggle/input/models/romanresner/uncertainty-aware-crop-perception/pytorch/default/18", 
    "/kaggle/working/project"
)

'/kaggle/working/project'

In [4]:
shutil.copytree(
    "/kaggle/input/models/romanresner/models-phnenobench-yolo11s-yolo26s/other/pt_onnx_engine/3",
    "/kaggle/working/project/models",
    dirs_exist_ok=True
)

'/kaggle/working/project/models'

In [4]:
yaml_content = """
path: /kaggle/input/notebooks/romanresner/phenobench-yolo-dataset/dataset_yolo


train: images/train
val: images/val
test: images/test

names:
  0: crop
  1: weed

task: segment
"""

with open("phenobench.yaml", "w") as f:
    f.write(yaml_content)

print("phenobench.yaml erstellt")

phenobench.yaml erstellt


In [ ]:
#%cd /kaggle/working/project
#!python main.py

In [5]:
import yaml

path = "/kaggle/working/project/configs/default.yaml"

with open(path, "r") as f:
    data = yaml.safe_load(f)

data["model"]["path"] = "/kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.engine"
data["io"]["input_video"] = "/kaggle/working/project/vertical_drone_flight.mp4"

data["io"]["output_video"] = "/kaggle/working/output/vertical_advanced_agricultural_engine.mp4"

data["io"]["metrics_output_csv"] = "/kaggle/working/output/run_metrics.csv"

data["io"]["metrics_output_json"] = "/kaggle/working/output/data/metrics.json"

data["io"]["max_frames"] = -1
data["io"]["side_by_side"] = True

with open(path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

In [14]:
# validate_accuracy.py
from ultralytics import YOLO
import torch
import gc
import json

# Clear GPU memory before starting
gc.collect()
torch.cuda.empty_cache()

# --- Shared Settings ----------------------------------------------------
DATA  = "/kaggle/working/phenobench.yaml"
IMGSZ = 960
BATCH = 4  # Lower to 2 or 1 if you encounter Out-Of-Memory (OOM) errors

# Define paths to all 8 model configurations (YOLO11s and YOLO26s)
models_config = {
    # YOLO11s Models
    "PT_11S_FP32":   "/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960.pt",
    "ONNX_11S_FP32": "/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960.onnx",
    "ONNX_11S_FP16": "/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960_fp16.onnx",
    "TRT_11S_FP32":  "/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960.engine",
    "TRT_11S_FP16":  "/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960_fp16.engine",
    
    # YOLO26s Models
    "PT_26S_FP32":   "/kaggle/working/project/models/phenobench_cropweed_seg_yolo26s_960.pt",
    "ONNX_26S_FP16": "/kaggle/working/project/models/phenobench_cropweed_seg_yolo26s_960_fp16.onnx",
    "TRT_26S_FP16": "/kaggle/working/project/models/phenobench_cropweed_seg_yolo26s_960_fp16.engine"

}

# Initialize results container
accuracy_results = {}

# --- Validation Loop ---------------------------------------------------
for name, path in models_config.items():
    print("=" * 102)
    print(f"Validating Model Accuracy: {name}")
    print("=" * 102)
    
    # Check if this is an FP16/half precision validation run (matches "FP16" in key name)
    is_half = "FP16" in name
    
    # Initialize and validate model
    model = YOLO(path, task="segment")
    metrics = model.val(data=DATA, imgsz=IMGSZ, split="val", device=0, batch=BATCH, half=is_half)
    
    # Store metrics for tabular display and graph generation
    accuracy_results[name] = {
        "box_map50": float(metrics.box.map50),
        "box_map": float(metrics.box.map),
        "seg_map50": float(metrics.seg.map50),
        "seg_map": float(metrics.seg.map)
    }
    
    # Clean up GPU memory after each run
    del model, metrics
    gc.collect()
    torch.cuda.empty_cache()

# --- Tabular Comparison Output -----------------------------------------
print("\n" + "=" * 102)
print("ACCURACY COMPARISON: YOLO11s (FP32/FP16) vs YOLO26s (FP32/FP16) across PT, ONNX, and TensorRT")
print("=" * 102)
print(f"{'Metric':<15} {'PT_11S_FP32':>12} {'ONNX_11S_FP32':>13} {'ONNX_11S_FP16':>13} {'TRT_11S_FP32':>12} {'TRT_11S_FP16':>12} {'PT_26S_FP32':>12} {'ONNX_26S_FP16':>13} {'TRT_26S_FP16':>12}")
print("-" * 102)

metrics_keys = [
    ("Box mAP50",    "box_map50"),
    ("Box mAP50-95", "box_map"),
    ("Seg mAP50",    "seg_map50"),
    ("Seg mAP50-95", "seg_map")
]

for label, key in metrics_keys:
    print(f"{label:<15} "
          f"{accuracy_results['PT_11S_FP32'][key]:>12.4f} "
          f"{accuracy_results['ONNX_11S_FP32'][key]:>13.4f} "
          f"{accuracy_results['ONNX_11S_FP16'][key]:>13.4f} "
          f"{accuracy_results['TRT_11S_FP32'][key]:>12.4f} "
          f"{accuracy_results['TRT_11S_FP16'][key]:>12.4f} "
          f"{accuracy_results['PT_26S_FP32'][key]:>12.4f} "
          f"{accuracy_results['ONNX_26S_FP16'][key]:>13.4f} "
          f"{accuracy_results['TRT_26S_FP16'][key]:>12.4f}")

print("=" * 102)

# Save metrics to a JSON file so that Script 2 can access them for plotting
metrics_json_path = "/kaggle/working/accuracy_metrics.json"
with open(metrics_json_path, "w") as f:
    json.dump(accuracy_results, f, indent=4)
print(f"Accuracy metrics successfully exported to: {metrics_json_path}")

Validating Model Accuracy: PT_11S_FP32
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s-seg summary (fused): 114 layers, 10,067,590 parameters, 0 gradients, 32.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2623.1±404.6 MB/s, size: 2632.6 KB)
val: Scanning /kaggle/input/notebooks/romanresner/phenobench-yolo-dataset/dataset_yolo/labels/val... 772 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 772/772 110.5it/s 7.0s<0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/notebooks/romanresner/phenobench-yolo-dataset/dataset_yolo/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 193/193 5.4it/s 35.6s0.2ss
                   all        772      10389      0.878      0.816      0.871      0.673      0.873      0.787      0.845      0.562
                  crop        772       6470      0.951     

In [16]:
# benchmark_performance.py
import yaml
import os
import csv
import json
import subprocess
import matplotlib.pyplot as plt

# --- 1. Define Paths and Settings --------------------------------------
PROJECT_DIR = "/kaggle/working/project"
CONFIG_PATH = "/kaggle/working/project/configs/default.yaml"

# Define configurations matching the 8 validation models
models_config = {
    # YOLO11s Models
    "PT_11S_FP32":   "/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960.pt",
    "ONNX_11S_FP32": "/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960.onnx",
    "ONNX_11S_FP16": "/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960_fp16.onnx",
    "TRT_11S_FP32":  "/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960.engine",
    "TRT_11S_FP16":  "/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960_fp16.engine",
    
    # YOLO26s Models
    "PT_26S_FP32":   "/kaggle/working/project/models/phenobench_cropweed_seg_yolo26s_960.pt",
    "ONNX_26S_FP16": "/kaggle/working/project/models/phenobench_cropweed_seg_yolo26s_960_fp16.onnx",
    "TRT_26S_FP16":  "/kaggle/working/project/models/phenobench_cropweed_seg_yolo26s_960_fp16.engine"
}

fps_results = {}
latency_results = {}


def get_metrics_from_csv(csv_path):
    """
    Reads the performance CSV file and calculates the average FPS
    and mean inference latency.
    """
    if not os.path.exists(csv_path):
        return 0.0, 0.0
    
    try:
        inference_times = []
        frame_times = []
        
        with open(csv_path, mode='r', newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                inference_times.append(float(row["inference_time_ms"]))
                frame_times.append(float(row["frame_time_ms"]))
        
        if not inference_times or not frame_times:
            return 0.0, 0.0
        
        # Calculate mean values from CSV rows
        mean_inference_time = sum(inference_times) / len(inference_times)
        mean_frame_time = sum(frame_times) / len(frame_times)
        mean_fps = 1000.0 / mean_frame_time if mean_frame_time > 0 else 0.0
        
        return mean_fps, mean_inference_time
    except Exception as e:
        print(f"Error reading CSV file {csv_path}: {e}")
        return 0.0, 0.0


# --- 2. Loop Over All 8 Models -----------------------------------------
for name, model_path in models_config.items():
    print("=" * 102)
    print(f"Configuring and starting video run for: {name}")
    print("=" * 102)
    
    # Read the configuration file (YAML)
    with open(CONFIG_PATH, "r") as f:
        data = yaml.safe_load(f)
    
    # Dynamically update YAML configuration values using the unified key
    data["model"]["path"] = model_path
    data["io"]["input_video"] = "/kaggle/working/project/vertical_drone_flight.mp4"
    data["io"]["output_video"] = f"/kaggle/working/output/vertical_advanced_agricultural_{name.lower()}.mp4"
    data["io"]["metrics_output_csv"] = f"/kaggle/working/output/data/{name.lower()}_run_metrics.csv"
    data["io"]["metrics_output_json"] = f"/kaggle/working/output/data/{name.lower()}_metrics.json"
    
    data["io"]["max_frames"] = -1
    data["io"]["side_by_side"] = True
    
    # Save the configuration file
    with open(CONFIG_PATH, "w") as f:
        yaml.dump(data, f, sort_keys=False)
    
    # Change to the project directory (equivalent to %cd)
    os.chdir(PROJECT_DIR)
    
    # Run main.py (equivalent to !python main.py)
    subprocess.run(["python", "main_profile.py"], check=True)
    
    # Calculate performance metrics directly from the newly created CSV file
    csv_path = f"/kaggle/working/output/data/{name.lower()}_run_metrics.csv"
    mean_fps, mean_latency = get_metrics_from_csv(csv_path)
    
    fps_results[name] = mean_fps
    latency_results[name] = mean_latency

# --- 3. Final Comparison Output of Performance Metrics -----------------
print("\n" + "=" * 102)
print("PERFORMANCE COMPARISON: Speed & Latency across PT, ONNX, and TensorRT")
print("=" * 102)
print(f"{'Metric':<15} {'PT_11S_FP32':>12} {'ONNX_11S_FP32':>13} {'ONNX_11S_FP16':>13} {'TRT_11S_FP32':>12} {'TRT_11S_FP16':>12} {'PT_26S_FP32':>12} {'ONNX_26S_FP16':>13} {'TRT_26S_FP16':>12}")
print("-" * 102)

print(f"{'Avg FPS':<15} "
      f"{fps_results['PT_11S_FP32']:>12.1f} "
      f"{fps_results['ONNX_11S_FP32']:>13.1f} "
      f"{fps_results['ONNX_11S_FP16']:>13.1f} "
      f"{fps_results['TRT_11S_FP32']:>12.1f} "
      f"{fps_results['TRT_11S_FP16']:>12.1f} "
      f"{fps_results['PT_26S_FP32']:>12.1f} "
      f"{fps_results['ONNX_26S_FP16']:>13.1f} "
      f"{fps_results['TRT_26S_FP16']:>12.1f}")

print(f"{'Latency (ms)':<15} "
      f"{latency_results['PT_11S_FP32']:>12.2f} "
      f"{latency_results['ONNX_11S_FP32']:>13.2f} "
      f"{latency_results['ONNX_11S_FP16']:>13.2f} "
      f"{latency_results['TRT_11S_FP32']:>12.2f} "
      f"{latency_results['TRT_11S_FP16']:>12.2f} "
      f"{latency_results['PT_26S_FP32']:>12.2f} "
      f"{latency_results['ONNX_26S_FP16']:>13.2f} "
      f"{latency_results['TRT_26S_FP16']:>12.2f}")
print("=" * 102)

# --- 4. Plotting Graphs ------------------------------------------------
model_names = list(models_config.keys())

# Graph 1: Processing Speed (FPS) Comparison
plt.figure(figsize=(11, 6))
fps_values = [fps_results[name] for name in model_names]

# Extended color list for all 8 configurations
colors = ['#2980b9', '#27ae60', '#2ecc71', '#d35400', '#e67e22', '#8e44ad', '#1abc9c', '#e84393']

bars = plt.bar(model_names, fps_values, color=colors, edgecolor='grey', width=0.55)
plt.title("Processing Speed Comparison (FPS)", fontsize=14, fontweight='bold')
plt.ylabel("Frames Per Second (FPS)", fontsize=12)
plt.xlabel("Model Configuration", fontsize=12)
plt.xticks(rotation=15, ha='right')  # Rotate labels slightly to prevent overlap
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add values above the bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height + 1.0, f"{height:.1f}", 
             ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
fps_plot_path = "/kaggle/working/fps_comparison.png"
plt.savefig(fps_plot_path, dpi=300)
plt.close()
print(f"FPS Comparison Plot saved to: {fps_plot_path}")

# Graph 2: Accuracy (mAP50) Comparison (Reading exported JSON data)
accuracy_json_path = "/kaggle/working/accuracy_metrics.json"
if os.path.exists(accuracy_json_path):
    try:
        with open(accuracy_json_path, "r") as f:
            accuracy_data = json.load(f)
            
        box_map50 = [accuracy_data[name]["box_map50"] for name in model_names]
        seg_map50 = [accuracy_data[name]["seg_map50"] for name in model_names]
        
        import numpy as np
        x_indices = np.arange(len(model_names))
        bar_width = 0.35
        
        plt.figure(figsize=(13, 6))
        rects1 = plt.bar(x_indices - bar_width/2, box_map50, bar_width, label='Box mAP50 (Detection)', color='#1abc9c')
        rects2 = plt.bar(x_indices + bar_width/2, seg_map50, bar_width, label='Seg mAP50 (Segmentation)', color='#3498db')
        
        plt.title("Model Accuracy Comparison (mAP50)", fontsize=14, fontweight='bold')
        plt.ylabel("mAP50 Score", fontsize=12)
        plt.xlabel("Model Configuration", fontsize=12)
        plt.xticks(x_indices, model_names, rotation=15, ha='right')  # Rotate labels
        plt.ylim(0.0, 1.1)
        plt.legend(loc='upper right')
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        
        plt.tight_layout()
        accuracy_plot_path = "/kaggle/working/map50_comparison.png"
        plt.savefig(accuracy_plot_path, dpi=300)
        plt.close()
        print(f"Accuracy Comparison Plot saved to: {accuracy_plot_path}")
    except Exception as e:
        print(f"Error generating accuracy plot: {e}")
else:
    print("Warning: 'accuracy_metrics.json' not found. Skip generating the accuracy plot.")
    print("Please run 'validate_accuracy.py' first to generate accuracy metrics.")

Configuring and starting video run for: PT_11S_FP32
PROFILING RUN
MAIN CUDA: True
MAIN GPU COUNT: 1
PYTHON: /usr/bin/python3
Starting Industrial Perception Pipeline with Profiling...
Info: NVML successfully initialized for GPU telemetry.
TRACKER CUDA: True
TRACKER GPU COUNT: 1
TRACKER DEVICE: 0
Loading model: /kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960.pt
Tracker initialized using: bytetrack.yaml
Processing full video: 708 frames.


100%|██████████| 708/708 [00:36<00:00, 19.38it/s]


📊 JSON Summary successfully compiled and saved to: /kaggle/working/output/data/pt_11s_fp32_run_metrics.json

Resources released.

PIPELINE STEP PROFILING SUMMARY (Average execution time per frame)
Pipeline Step                              Avg Time (ms)       Percentage (%)
-----------------------------------------------------------------------------------
video_read                                        1.27 ms                2.5%
perception_inference                             48.58 ms               94.3%
state_memory_update                               0.09 ms                0.2%
decision_engine                                   0.03 ms                0.1%
gpu_telemetry                                     1.38 ms                2.7%
telemetry_logging                                 0.17 ms                0.3%
-----------------------------------------------------------------------------------
Total Pipeline Latency                           51.53 ms              100.0%
Calculated 

100%|██████████| 708/708 [00:50<00:00, 13.88it/s]


📊 JSON Summary successfully compiled and saved to: /kaggle/working/output/data/onnx_11s_fp32_run_metrics.json

Resources released.

PIPELINE STEP PROFILING SUMMARY (Average execution time per frame)
Pipeline Step                              Avg Time (ms)       Percentage (%)
-----------------------------------------------------------------------------------
video_read                                        1.35 ms                1.9%
perception_inference                             68.94 ms               95.8%
state_memory_update                               0.13 ms                0.2%
decision_engine                                   0.03 ms                0.0%
gpu_telemetry                                     1.30 ms                1.8%
telemetry_logging                                 0.17 ms                0.2%
-----------------------------------------------------------------------------------
Total Pipeline Latency                           71.93 ms              100.0%
Calculate

...

📊 JSON Summary successfully compiled and saved to: /kaggle/working/output/data/onnx_26s_fp16_run_metrics.json

Resources released.

PIPELINE STEP PROFILING SUMMARY (Average execution time per frame)
Pipeline Step                              Avg Time (ms)       Percentage (%)
-----------------------------------------------------------------------------------
video_read                                        1.29 ms                2.4%
perception_inference                             50.87 ms               94.7%
state_memory_update                               0.09 ms                0.2%
decision_engine                                   0.03 ms                0.1%
gpu_telemetry                                     1.28 ms                2.4%
telemetry_logging                                 0.17 ms                0.3%
-----------------------------------------------------------------------------------
Total Pipeline Latency                           53.73 ms              100.0%
Calculate

  0%|          | 0/708 [00:00<?, ?it/s]

[06/06/2026-14:50:43] [TRT] [I] Loaded engine size: 22 MiB
[06/06/2026-14:50:43] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +48, now: CPU 0, GPU 71 (MiB)


100%|██████████| 708/708 [00:32<00:00, 21.82it/s]


📊 JSON Summary successfully compiled and saved to: /kaggle/working/output/data/trt_26s_fp16_run_metrics.json

Resources released.

PIPELINE STEP PROFILING SUMMARY (Average execution time per frame)
Pipeline Step                              Avg Time (ms)       Percentage (%)
-----------------------------------------------------------------------------------
video_read                                        1.29 ms                2.8%
perception_inference                             42.70 ms               93.3%
state_memory_update                               0.10 ms                0.2%
decision_engine                                   0.03 ms                0.1%
gpu_telemetry                                     1.48 ms                3.2%
telemetry_logging                                 0.17 ms                0.4%
-----------------------------------------------------------------------------------
Total Pipeline Latency                           45.77 ms              100.0%
Calculated

In [25]:
%cd /kaggle/working/project
!python main_profile.py

/kaggle/working/project
PROFILING RUN
MAIN CUDA: True
MAIN GPU COUNT: 1
PYTHON: /usr/bin/python3
Starting Industrial Perception Pipeline with Profiling...
Info: NVML successfully initialized for GPU telemetry.
TRACKER CUDA: True
TRACKER GPU COUNT: 1
TRACKER DEVICE: 0
Loading model: /kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.engine
Tracker initialized using: bytetrack.yaml
Processing full video: 708 frames.
  0%|                                                   | 0/708 [00:00<?, ?it/s]Loading /kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.engine for TensorRT inference...
[06/06/2026-09:32:08] [TRT] [I] Loaded engine size: 22 MiB
[06/06/2026-09:32:08] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +48, now: CPU 0, GPU 71 (MiB)
100%|█████████████████████████████████████████| 708/708 [00:30<00:00, 23.34it/s]
📊 JSON Summary successfully compiled and saved to: /kaggle/working/output/run_metrics.json

Resources re

In [7]:
from ultralytics import YOLO

# ONNX direkt in FP16 exportieren
model = YOLO("/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960.pt")

             #/kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.pt
model.export(
    format="onnx",
    imgsz=960,
    half=True,    # ← FP16 Gewichte direkt im ONNX
    device=0,
    task="segment"
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s-seg summary (fused): 114 layers, 10,067,590 parameters, 0 gradients, 32.8 GFLOPs

PyTorch: starting from '/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960.pt' with input shape (1, 3, 960, 960) BCHW and output shape(s) ((1, 38, 18900), (1, 32, 240, 240)) (19.6 MB)
requirements: Ultralytics requirements ['onnxruntime-gpu', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 364ms
Prepared 2 packages in 2.91s
Installed 2 packages in 20ms
 + onnxruntime-gpu==1.26.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 3.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 18...
ONNX: slimming

'/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960.onnx'

In [9]:
import tensorrt as trt
import json, os

# FP16 ONNX verwenden
ONNX_PATH   = "/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960_fp16.onnx"  # half=True exportiert
ENGINE_PATH = "/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960_fp16.engine"

METADATA = {
    "task": "segment",
    "imgsz": [960, 960],
    "nc": 2,
    "names": {0: "Crop", 1: "Weed"}
}

logger = trt.Logger(trt.Logger.WARNING)

with trt.Builder(logger) as builder, \
     builder.create_network() as network, \
     trt.OnnxParser(network, logger) as parser, \
     builder.create_builder_config() as config:

    with open(ONNX_PATH, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            raise RuntimeError("ONNX parsing failed")

    # Kein Flag nötig — ONNX hat bereits FP16 Typen
    # TRT liest DataType.HALF direkt aus dem ONNX Graph
    engine_bytes = builder.build_serialized_network(network, config)
    if engine_bytes is None:
        raise RuntimeError("Engine build failed")

    with open(ENGINE_PATH, "wb") as f:
        meta = json.dumps(METADATA)
        f.write(len(meta).to_bytes(4, byteorder="little", signed=True))
        f.write(meta.encode())
        f.write(engine_bytes)

print(f"FP16 Engine gespeichert: {os.path.getsize(ENGINE_PATH)/1024/1024:.1f} MB")

✅ FP16 Engine gespeichert: 21.7 MB


In [14]:
#YOLO26
import tensorrt as trt
import json, os
from ultralytics import YOLO

pt_model = YOLO("/kaggle/working/models/best.pt")

METADATA = {
    "task":        "segment",
    "imgsz":       [960, 960],
    "nc":          int(pt_model.model.nc),
    "names":       {int(k): v for k, v in pt_model.model.names.items()},
    "stride":      int(max(pt_model.model.stride)),
    "batch":       1,
}

print("METADATA:", json.dumps(METADATA, indent=2))

# FP16 ONNX verwenden
ONNX_PATH   = "/kaggle/working/models/best.onnx"  # half=True exportiert
ENGINE_PATH = "/kaggle/working/models/best.engine"
"""
METADATA = {
    "task": "segment",
    "imgsz": [960, 960],
    "nc": 2,
    "names": {0: "Crop", 1: "Weed"}
}
"""
logger = trt.Logger(trt.Logger.WARNING)

with trt.Builder(logger) as builder, \
     builder.create_network() as network, \
     trt.OnnxParser(network, logger) as parser, \
     builder.create_builder_config() as config:

    with open(ONNX_PATH, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            raise RuntimeError("ONNX parsing failed")

    # Kein Flag nötig — ONNX hat bereits FP16 Typen
    # TRT liest DataType.HALF direkt aus dem ONNX Graph
    engine_bytes = builder.build_serialized_network(network, config)
    if engine_bytes is None:
        raise RuntimeError("Engine build failed")

    with open(ENGINE_PATH, "wb") as f:
        meta = json.dumps(METADATA)
        f.write(len(meta).to_bytes(4, byteorder="little", signed=True))
        f.write(meta.encode())
        f.write(engine_bytes)

print(f"✅ FP16 Engine gespeichert: {os.path.getsize(ENGINE_PATH)/1024/1024:.1f} MB")

METADATA: {
  "task": "segment",
  "imgsz": [
    960,
    960
  ],
  "nc": 2,
  "names": {
    "0": "crop",
    "1": "weed"
  },
  "stride": 32,
  "batch": 1
}
[06/05/2026-21:11:08] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
✅ FP16 Engine gespeichert: 22.3 MB


In [17]:
from ultralytics import YOLO
model = YOLO("/kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.pt")
model.model.model[-1].end2end = False  # Segment26 Head
model.export(
    format="onnx",
    imgsz=960,
    half=True,
    device=0,
    task="segment",
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s-seg summary (fused): 175 layers, 11,415,062 parameters, 0 gradients, 34.1 GFLOPs

PyTorch: starting from '/kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.pt' with input shape (1, 3, 960, 960) BCHW and output shape(s) ((1, 38, 18900), (1, 32, 240, 240)) (22.3 MB)

ONNX: starting export with onnx 1.21.0 opset 18...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 1.8s, saved as '/kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.onnx' (20.1 MB)

Export complete (2.3s)
Results saved to /kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.onnx
Predict:         yolo predict task=segment model=/kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.onnx imgsz=960 half
Validate:        yolo val task=segment model=/kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.onnx imgsz=960 data=/kaggle/working/data.yaml half 
Visualize:       https://netron.app


'/kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.onnx'

In [16]:
import onnx
m = onnx.load("/kaggle/working/models/phenobench_cropweed_seg_yolo26s_960_fp16.onnx")
for output in m.graph.output:
    shape = [d.dim_value for d in output.type.tensor_type.shape.dim]
    print(output.name, shape)

output0 [1, 38, 18900]
output1 [1, 32, 240, 240]


In [18]:
import tensorrt as trt
import json, os
from ultralytics import YOLO

# Metadata direkt aus .pt lesen
pt_model = YOLO("/kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.pt")

METADATA = {
    "task":   "segment",
    "imgsz":  [960, 960],
    "nc":     int(pt_model.model.nc),
    "names":  {int(k): v for k, v in pt_model.model.names.items()},
    "stride": int(max(pt_model.model.stride)),
    "batch":  1,
}
print("METADATA:", json.dumps(METADATA, indent=2))
del pt_model  # GPU Speicher freigeben

ONNX_PATH   = "/kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.onnx"
ENGINE_PATH = "/kaggle/working/models/phenobench_cropweed_seg_yolo26s_960.engine"

logger = trt.Logger(trt.Logger.WARNING)

with trt.Builder(logger) as builder, \
     builder.create_network() as network, \
     trt.OnnxParser(network, logger) as parser, \
     builder.create_builder_config() as config:

    with open(ONNX_PATH, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            raise RuntimeError("ONNX parsing failed")

    # Verify shapes before building
    print(f"Network inputs:  {network.num_inputs}")
    print(f"Network outputs: {network.num_outputs}")
    for i in range(network.num_outputs):
        out = network.get_output(i)
        print(f"  output{i}: {out.shape}")

    engine_bytes = builder.build_serialized_network(network, config)
    if engine_bytes is None:
        raise RuntimeError("Engine build failed")

    with open(ENGINE_PATH, "wb") as f:
        meta = json.dumps(METADATA)
        f.write(len(meta).to_bytes(4, byteorder="little", signed=True))
        f.write(meta.encode())
        f.write(engine_bytes)

print(f"✅ Engine: {os.path.getsize(ENGINE_PATH)/1024/1024:.1f} MB")

METADATA: {
  "task": "segment",
  "imgsz": [
    960,
    960
  ],
  "nc": 2,
  "names": {
    "0": "crop",
    "1": "weed"
  },
  "stride": 32,
  "batch": 1
}
Network inputs:  1
Network outputs: 2
  output0: (1, 38, 18900)
  output1: (1, 32, 240, 240)
✅ Engine: 22.4 MB


In [20]:
import os
import csv
import json
import matplotlib.pyplot as plt

# --- 1. Define Paths and Configurations ---------------------------------
ACCURACY_JSON = "/kaggle/working/accuracy_metrics.json"
CSV_DIR = "/kaggle/working/output/data"

# The 8 models used in previous runs
models = [
    "PT_11S_FP32", "ONNX_11S_FP32", "ONNX_11S_FP16", "TRT_11S_FP32", "TRT_11S_FP16",
    "PT_26S_FP32", "ONNX_26S_FP16", "TRT_26S_FP16"
]

# Color mapping for each model configuration to keep plots consistent
model_colors = {
    "PT_11S_FP32": '#2980b9',   # Blue
    "ONNX_11S_FP32": '#27ae60', # Dark Green
    "ONNX_11S_FP16": '#2ecc71', # Light Green
    "TRT_11S_FP32": '#d35400',  # Dark Orange
    "TRT_11S_FP16": '#e67e22',  # Light Orange
    "PT_26S_FP32": '#8e44ad',   # Purple
    "ONNX_26S_FP16": '#1abc9c', # Teal
    "TRT_26S_FP16": '#e84393'   # Magenta
}

# Specific text offsets for each model to prevent overlapping labels.
# Only TRT_11S_FP32 goes below, while all other models stay on top.
model_offsets = {
    "PT_11S_FP32": (8, 5),       # Stays on top
    "ONNX_11S_FP32": (8, 5),     # Stays on top
    "ONNX_11S_FP16": (8, 5),     # Stays on top
    "TRT_11S_FP32": (8, -12),    # Positioned below (TRT_11S_FP32 unten)
    "TRT_11S_FP16": (8, 5),      # Stays on top
    "PT_26S_FP32": (8, 5),       # Stays on top
    "ONNX_26S_FP16": (8, 5),     # Stays on top
    "TRT_26S_FP16": (8, 5)       # Stays on top
}


def get_fps_from_csv(csv_path):
    """Reads the CSV file and returns the average FPS."""
    if not os.path.exists(csv_path):
        return None
    try:
        frame_times = []
        with open(csv_path, mode='r', newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                frame_times.append(float(row["frame_time_ms"]))
        if not frame_times:
            return None
        mean_frame_time = sum(frame_times) / len(frame_times)
        mean_fps = 1000.0 / mean_frame_time if mean_frame_time > 0 else 0.0
        return mean_fps
    except Exception as e:
        print(f"Error reading {csv_path}: {e}")
        return None


# --- 2. Load Data ------------------------------------------------------
if not os.path.exists(ACCURACY_JSON):
    raise FileNotFoundError(
        f"Could not find '{ACCURACY_JSON}'. "
        "Please make sure you have run the validation script first."
    )

with open(ACCURACY_JSON, "r") as f:
    accuracy_data = json.load(f)

fps_data = {}
for name in models:
    csv_path = f"{CSV_DIR}/{name.lower()}_run_metrics.csv"
    fps = get_fps_from_csv(csv_path)
    if fps is not None:
        fps_data[name] = fps
    else:
        print(f"Warning: Could not read FPS for {name} from {csv_path}")
        fps_data[name] = 0.0


# --- 3. Plotting Function with Requested Offsets -----------------------
def generate_scatter_plot(metric_key, title, ylabel, filename):
    """Generates a compact scatter plot of a specific metric vs FPS with labeled points."""
    plt.figure(figsize=(6, 5))
    
    x_vals = []
    y_vals = []
    
    # Plot each model configuration as a labeled point
    for name in models:
        fps = fps_data[name]
        metric_val = accuracy_data[name][metric_key]
        
        x_vals.append(fps)
        y_vals.append(metric_val)
        
        color = model_colors.get(name, '#7f8c8d')
        
        # Draw the scatter point
        plt.scatter(fps, metric_val, color=color, s=100, edgecolors='black', zorder=5, label=name)
        
        # Retrieve the custom top/bottom offset for this model
        offset = model_offsets.get(name, (8, 4))
        
        plt.annotate(
            name, 
            (fps, metric_val), 
            textcoords="offset points", 
            xytext=offset, 
            ha='left', 
            fontsize=8, 
            fontweight='bold',
            zorder=6
        )
        
    plt.title(title, fontsize=11, fontweight='bold')
    plt.xlabel("Speed (FPS)", fontsize=10)
    plt.ylabel(ylabel, fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # Calculate dynamic, tight margins based on the data range
    if x_vals:
        x_min, x_max = min(x_vals), max(x_vals)
        x_range = x_max - x_min if x_max != x_min else 10.0
        # Small padding on the left, slightly more on the right for text labels
        plt.xlim(x_min - x_range * 0.05, x_max + x_range * 0.20)
        
    if y_vals:
        y_min, y_max = min(y_vals), max(y_vals)
        y_range = y_max - y_min if y_max != y_min else 0.1
        # Tight padding on the y-axis, bounded between 0.0 and 1.0
        plt.ylim(max(0.0, y_min - y_range * 0.15), min(1.0, y_max + y_range * 0.15))
        
    plt.tight_layout()
    save_path = f"/kaggle/working/{filename}"
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Plot successfully generated and saved to: {save_path}")


# --- 4. Generate the 4 Required Plots ----------------------------------
# Plot 1: Box mAP50 vs FPS
generate_scatter_plot(
    metric_key="box_map50",
    title="Detection Accuracy (Box mAP50) vs Speed",
    ylabel="Box mAP50",
    filename="box_map50_vs_fps.png"
)

# Plot 2: Box mAP50-95 vs FPS
generate_scatter_plot(
    metric_key="box_map",
    title="Detection Accuracy (Box mAP50-95) vs Speed",
    ylabel="Box mAP50-95",
    filename="box_map50_95_vs_fps.png"
)

# Plot 3: Seg mAP50 vs FPS
generate_scatter_plot(
    metric_key="seg_map50",
    title="Segmentation Accuracy (Seg mAP50) vs Speed",
    ylabel="Seg mAP50",
    filename="seg_map50_vs_fps.png"
)

# Plot 4: Seg mAP50-95 vs FPS
generate_scatter_plot(
    metric_key="seg_map",
    title="Segmentation Accuracy (Seg mAP50-95) vs Speed",
    ylabel="Seg mAP50-95",
    filename="seg_map50_95_vs_fps.png"
)

print("\nAll 4 compact plots for 8 models have been created.")

Plot successfully generated and saved to: /kaggle/working/box_map50_vs_fps.png
Plot successfully generated and saved to: /kaggle/working/box_map50_95_vs_fps.png
Plot successfully generated and saved to: /kaggle/working/seg_map50_vs_fps.png
Plot successfully generated and saved to: /kaggle/working/seg_map50_95_vs_fps.png

All 4 compact plots for 8 models have been created.


In [21]:
import os
import zipfile
from IPython.display import FileLink

# --- 1. Define Files and Directories to Archive ------------------------
files_to_zip = [
    "/kaggle/working/box_map50_vs_fps.png",
    "/kaggle/working/box_map50_95_vs_fps.png",
    "/kaggle/working/seg_map50_vs_fps.png",
    "/kaggle/working/seg_map50_95_vs_fps.png",
    "/kaggle/working/map50_comparison.png",
    "/kaggle/working/fps_comparison.png",
    "/kaggle/working/accuracy_metrics.json"
]

output_dir = "/kaggle/working/output"
zip_filename = "/kaggle/working/benchmark_results.zip"

# --- 2. Create the ZIP Archive -----------------------------------------
print("Creating ZIP archive...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add individual image and JSON files
    for file_path in files_to_zip:
        if os.path.exists(file_path):
            # Write file and place it directly in the root of the ZIP
            zipf.write(file_path, os.path.basename(file_path))
            print(f"Added: {os.path.basename(file_path)}")
        else:
            print(f"Warning: File not found: {file_path}")
            
    # Add the entire output directory recursively, preserving folder structure
    if os.path.exists(output_dir):
        for root, dirs, files in os.walk(output_dir):
            for file in files:
                file_path = os.path.join(root, file)
                # Determine the relative path to maintain directory structure inside ZIP
                relative_path = os.path.relpath(file_path, start="/kaggle/working")
                zipf.write(file_path, relative_path)
        print(f"Added folder: {output_dir}")
    else:
        print(f"Warning: Output directory not found: {output_dir}")

print(f"\nArchive successfully compiled: {zip_filename}")
print("=" * 73)
print("Click the link below to download your benchmark results:")
print("=" * 73)

# --- 3. Generate Clickable Download Link -------------------------------
# This generates an HTML link directly in your notebook output cell.
# You can also download the ZIP from the right sidebar in the Kaggle UI.
FileLink(r'benchmark_results.zip')

Creating ZIP archive...
Added: box_map50_vs_fps.png
Added: box_map50_95_vs_fps.png
Added: seg_map50_vs_fps.png
Added: seg_map50_95_vs_fps.png
Added: map50_comparison.png
Added: fps_comparison.png
Added: accuracy_metrics.json
Added folder: /kaggle/working/output

Archive successfully compiled: /kaggle/working/benchmark_results.zip
Click the link below to download your benchmark results:


/kaggle/working/project/benchmark_results.zip

In [6]:
from ultralytics import YOLO
model = YOLO("/kaggle/working/project/models/phenobench_cropweed_seg_yolo26s_960.pt")
model.info()


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO26s-seg summary: 309 layers, 11,435,366 parameters, 0 gradients, 37.1 GFLOPs


(309, 11435366, 0, 37.085593599999996)

In [7]:
model = YOLO("/kaggle/working/project/models/phenobench_cropweed_seg_yolo11s_960.pt")
model.info()

YOLO11s-seg summary: 204 layers, 10,083,062 parameters, 0 gradients, 33.1 GFLOPs


(204, 10083062, 0, 33.0778112)